# 03 · Encapsulation: Private Attributes, Getters/Setters & `@property`

**Module 06 – Object-Oriented Programming** · Estimated time: 45 min

> 📘 The handbook covers this topic twice (OOP chapter + Advanced chapter); both are merged here. 📗 The bootcamp doesn't cover `@property`.

### 🎯 What you will learn
- Why control access to attributes (validation, read-only, side effects, flexibility)
- Python's privacy conventions: `_single` and `__double` (name mangling)
- Traditional getter/setter methods
- ✅ The Pythonic way: `@property`, `@x.setter`, `@x.deleter`
- Read-only and computed properties
- `functools.cached_property`

## 1. Why getters and setters? (📘)

- **Validation**: refuse invalid values (e.g. a negative age)
- **Read-only attributes**: allow reading but not changing
- **Side effects**: log or update something whenever a value changes
- **Maintainability**: change the internal storage without breaking code that uses the class

## 2. "Private" attributes in Python

Python has **no truly private attributes** (unlike Java/C++). It uses **conventions**:

| Name | Meaning |
|---|---|
| `name` | public |
| `_name` | "internal, please don't touch" (a strong convention, like a *Please Do Not Touch* sign) |
| `__name` | **name mangling**: renamed to `_ClassName__name` to avoid clashes in subclasses |

In [1]:
class MyClass:
    def __init__(self):
        self._internal_value = 0       # convention: _ means "private"
        self.__secret = "mangled"      # name-mangled

    def get_value(self):
        return self._internal_value

obj = MyClass()
print(obj.get_value())                 # ✅ the preferred way
print(obj._internal_value)             # works, but against convention
print(obj._MyClass__secret)            # mangled name: still reachable, just hidden

0
0
mangled


In [2]:
obj.__secret                           # AttributeError: the name was mangled

AttributeError: 'MyClass' object has no attribute '__secret'

> 💡 Use `_single_underscore` for internal attributes. Use `__double` only when you need to avoid name clashes with subclasses.

## 3. Traditional getters and setters (📘)

In [3]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self._age = age

    def get_age(self):                       # getter
        return self._age

    def set_age(self, new_age):              # setter with validation
        if 0 <= new_age <= 150:
            self._age = new_age
        else:
            print("Invalid age!")

person = Person("Alice", 30)
print(person.get_age())
person.set_age(35)
print(person.get_age())
person.set_age(-5)
print(person.get_age())                      # unchanged

30
35
Invalid age!
35


This works, but `p.set_age(p.get_age() + 1)` is clunky, and it's the Java style. Python has something better.

## 4. ✅ The Pythonic way: `@property` (📘)

`@property` lets you use **attribute syntax** (`p.age`, `p.age = 5`) while running getter/setter code behind the scenes.

In [4]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age               # ❗ goes through the setter → validated even at creation

    @property
    def age(self):                   # getter
        return self._age

    @age.setter
    def age(self, new_age):          # setter
        if not 0 <= new_age <= 150:
            raise ValueError(f"Invalid age: {new_age}")
        self._age = new_age

    @age.deleter
    def age(self):                   # deleter
        print("Deleting age")
        del self._age

person = Person("Bob", 40)
print(person.age)            # calls the getter
person.age = 45              # calls the setter
print(person.age)
try:
    person.age = -22
except ValueError as e:
    print("ValueError:", e)
del person.age               # calls the deleter

40
45
ValueError: Invalid age: -22
Deleting age


**Benefits of `@property`:**
- Clean attribute-like syntax: `obj.name` instead of `obj.get_name()`
- **Consistent interface**: start with a plain attribute; later turn it into a property (adding validation) without changing any calling code
- Read-only properties by omitting the setter

> 💡 Don't write getters/setters "just in case". In Python, start with plain public attributes; convert to a property only when you need logic.

## 5. Read-only and computed properties (📘)

In [5]:
class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def radius(self):              # read-only: no setter
        return self._radius

    @property
    def area(self):                # computed on every access
        return 3.1416 * self._radius ** 2

    @property
    def diameter(self):
        return self._radius * 2

c = Circle(5)
print(c.radius, c.area, c.diameter)

5 78.53999999999999 10


In [6]:
c.area = 20          # AttributeError: property 'area' of 'Circle' object has no setter

AttributeError: property 'area' of 'Circle' object has no setter

## 6. A realistic example: temperature with unit conversion

In [7]:
class Temperature:
    def __init__(self, celsius=0.0):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Below absolute zero!")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = (value - 32) * 5 / 9    # reuse the validation

t = Temperature(25)
print(t.celsius, t.fahrenheit)
t.fahrenheit = 212
print(t.celsius)

25 77.0
100.0


Internally we only store Celsius, yet users can read and write either unit. That's the **maintainability** benefit from the handbook in action.

## 7. `functools.cached_property`: compute once

For expensive computed values that don't change:

In [8]:
from functools import cached_property
import time

class Dataset:
    def __init__(self, values):
        self.values = values

    @cached_property
    def stats(self):
        print("Computing stats (slow)...")
        time.sleep(0.2)
        return {"min": min(self.values), "max": max(self.values),
                "mean": sum(self.values) / len(self.values)}

ds = Dataset([3, 8, 1, 9])
print(ds.stats)      # computed
print(ds.stats)      # cached: no recomputation

Computing stats (slow)...


{'min': 1, 'max': 9, 'mean': 5.25}
{'min': 1, 'max': 9, 'mean': 5.25}


## 8. Encapsulation in practice: a bank account

In [9]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance
        self._history = []

    @property
    def balance(self):                 # read-only from outside
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self._balance += amount
        self._history.append(+amount)

    def withdraw(self, amount):
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount
        self._history.append(-amount)

acct = BankAccount("Asha", 100)
acct.deposit(50)
acct.withdraw(30)
print(acct.balance)
try:
    acct.balance = 1_000_000           # can't cheat!
except AttributeError as e:
    print("AttributeError:", e)

120
AttributeError: property 'balance' of 'BankAccount' object has no setter


## 📝 Summary
- No real privacy in Python: `_x` = internal by convention, `__x` = name-mangled.
- Getters/setters enable validation, read-only access and flexibility.
- ✅ Use `@property` (+ `.setter` / `.deleter`) instead of `get_x()`/`set_x()`.
- Omit the setter for read-only; compute values in a property; `@cached_property` to cache.

## 🏋️ Practice
1. Create `Product` with a `price` property that rejects negative values, and a read-only `price_with_tax` (18%).
2. Create `Rectangle` with `width` and `height` properties (must be > 0) and a computed `area`.
3. Create `User` with an `email` property that lower-cases and validates the value contains `@`.

<details><summary>Show solutions</summary>

```python
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price
    @property
    def price(self): return self._price
    @price.setter
    def price(self, v):
        if v < 0: raise ValueError("negative price")
        self._price = v
    @property
    def price_with_tax(self): return round(self._price * 1.18, 2)

class Rectangle:
    def __init__(self, w, h): self.width, self.height = w, h
    @property
    def width(self): return self._w
    @width.setter
    def width(self, v):
        if v <= 0: raise ValueError
        self._w = v
    @property
    def height(self): return self._h
    @height.setter
    def height(self, v):
        if v <= 0: raise ValueError
        self._h = v
    @property
    def area(self): return self._w * self._h

class User:
    def __init__(self, email): self.email = email
    @property
    def email(self): return self._email
    @email.setter
    def email(self, v):
        if "@" not in v: raise ValueError("invalid email")
        self._email = v.strip().lower()
```
</details>